In [4]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [5]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_1(x,n):
    y = 1-(1-x)**n
    return(y)

def h_2(x,r):
    y = (1+r)*x-r*x**2
    return(y)

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def h_4(x,r):
    if x < 1/2:
        return((1+r)*x)
    else:
        return((1-r)*p+r)

In [6]:
def Robust_exp_h1 (x,p,r,m):
    N = len(x)
    q = cp.Variable(N)
    q_b = cp.Variable(N)
    phi_cons = 0
    psets = list(powerset(list(range(N))))
    for i in range(1,len(psets)):
        psets[i] = list(psets[i])
    psets = psets[1:(len(psets)-1)]
    for i in range(N):
        phi_cons = phi_cons -(cp.entr(q[i]) + q[i]*np.log(p[i]))
    constraints = [q >= 0, q_b >= 0, cp.sum(q) == 1, cp.sum(q_b) == 1, phi_cons <= r]
    for i in range(len(psets)):
        z1 = q[psets[i]]
        z2 = q_b[psets[i]]
        constraints.append(cp.sum(z2)-(1-(1-cp.sum(z1))**m) <= 0)
    obj = cp.Minimize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve()
    return(prob.value,q.value,q_b.value)


def Robust_dual_h1 (x, p, r, m):
    N = len(x)
    lbda = cp.Variable(N)
    v = cp.Variable((N,N))
    alpha = cp.Variable(1)
    gamma = cp.Variable(1)
    c = 1/(m**(1/(m-1)))+1/(m**(m/(m-1)))
    z2 = 0
    constraints = [gamma >= 0, lbda >= 0]
    for i in range(N):
        for j in range((N-i)):
            constraints.append(v[i,j] <= 0)
        constraints.append(-x[i]-cp.sum(lbda[(N-i):N]))
    for j in range(N):
        z1 = -cp.min(v[j,(N+1-j):N])
        z2 = z2 + z1 + c*z1**(m/(m-1))*1/(lbda[j]**(1/(m-1))) + lbda[j]
    v = 1/(alpha)**(1/3)
    print(v.curvature)
    print(z2.curvature)
    return(0) 

In [63]:
def Robust_exp_h3 (x,p,r,m):
    N = len(x)
    q = cp.Variable(N)
    q_b = cp.Variable(N)
    phi_cons = 0
    psets = list(powerset(list(range(N))))
    for i in range(1,len(psets)):
        psets[i] = list(psets[i])
    psets = psets[1:(len(psets)-1)]
    for i in range(N):
        phi_cons = phi_cons -(cp.entr(q[i]) + q[i]*np.log(p[i]))
    constraints = [q >= 0, q_b >= 0, cp.sum(q) == 1, cp.sum(q_b) == 1, phi_cons <= r]
    for i in range(len(psets)):
        z1 = q[psets[i]]
        z2 = q_b[psets[i]]
        v= -cp.neg(cp.sum(z1)/(1-m)-1)+1
        constraints.append(cp.sum(z2)-v <= 0)
    obj = cp.Maximize(-q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve()
    return(prob.value,q.value,q_b.value)  


def Robust_dual_h3 (x, p, r, m):
    N = len(x)
    lbda = cp.Variable(N)
    v = cp.Variable((N,N))
    alpha = cp.Variable(1)
    gamma = cp.Variable(1)
    z2 = 0
    z4 = 0
    constraints = [gamma >= 0, lbda >= 0]
    for i in range(N-1):
        constraints.append(v[i][0:(N-i-1)] <= 0)
        constraints.append(v[i][(N-i-1):N] >= 0)
        constraints.append(-x[i]-cp.sum(lbda[(N-i):N]))
    constraints.append(v[N-1] >= 0)
    for j in range(N):
        z1 = -cp.min(v[j,(N+1-j):N])*(1-m)
        z2 = z2 + cp.pos(z1)
    for i in range(N):
        z3 = (-alpha + cp.sum(v[0:N:1,i]))/gamma
        print(z3.curvature)
    return(0) 

In [64]:
x=np.array([1,2,3,4])
p=np.array([0.2,0.4,0.3,0.1])
r=1
m=0.5
Robust_dual_h3(x,p,r,m)

UNKNOWN
UNKNOWN
UNKNOWN
UNKNOWN


0

In [25]:
N=4
x=np.array([1,2,3])
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets)-1)]
psets

[[0],
 [1],
 [2],
 [3],
 [0, 1],
 [0, 2],
 [0, 3],
 [1, 2],
 [1, 3],
 [2, 3],
 [0, 1, 2],
 [0, 1, 3],
 [0, 2, 3],
 [1, 2, 3]]

In [31]:
for i in range(len(psets)):
    print(sum(qbvalue[psets[i]])-h_3(sum(qvalue[psets[i]]),m))
print(qbvalue)

-9.665501732314397e-11
-0.3206046912235386
-0.3332749253456948
-0.2874020055775485
-1.694866469392764e-12
-1.0815681683595813e-10
-1.4244871948676519e-10
-0.6538796165692333
-0.6080066968010871
-0.6206769309232433
-1.3196665982206923e-11
-4.7488568633013983e-11
-1.5395051899957934e-10
-0.9412816221467819
[ 1.00000000e+00  9.49601647e-11 -1.15017497e-11 -4.57936792e-11]


In [7]:
print(qbvalue)
print(qvalue)

[9.99574480e-01 4.10014239e-04 1.54870993e-05 1.39706767e-08]
[0.85637513 0.08091854 0.04707461 0.01563172]


In [62]:
x=np.array([[0,1,2,3],[7,8,9,6]])
x[0:2:1,1]

array([1, 8])